# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StickRift/Rift/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

My lane is Refresh / Content Opportunity Scoring. The goal is to identify content pages that may deserve human review for a possible refresh.

My rule will prioritize pages that have not been updated recently but still receive meaningful search visibility. These pages may be worth reviewing because they continue to attract impressions despite being potentially outdated.

### Signal checks

I will examine content staleness and search impressions using bucket tables that show the number of pages in each group. I will use the observed results to judge whether the signals support my rule before finalizing its thresholds.

### Score, reason code, and action

The baseline will use one transparent rule and one reason code. Pages meeting the selected conditions will receive a higher score and the action label “Review for content refresh.”

The score is a decision-support ranking, not proof that refreshing a page will improve its performance. Each recommendation requires human review.


In [10]:
import pandas as pd
import numpy as np

# Load the Content Refresh dataset directly from GitHub
data_url = "https://raw.githubusercontent.com/StickRift/Rift/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_url)

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [11]:
# Inspect the two selected signals

print("DAYS SINCE LAST UPDATE")
print(df["days_since_last_update"].describe())

print("\nIMPRESSIONS OVER 90 DAYS")
print(df["impressions_90d"].describe())

print("\nTREND DIRECTION")
print(df["trend_direction"].value_counts(dropna=False))

DAYS SINCE LAST UPDATE
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

IMPRESSIONS OVER 90 DAYS
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

TREND DIRECTION
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [12]:
# Create buckets for the two selected signals

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, float("inf")],
    labels=["0-30 days", "31-90 days", "Over 90 days"]
)

df["impressions_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 100, 1000, float("inf")],
    labels=["Low", "Medium", "High"]
)

# Signal 1: Staleness versus observed trend
print("SIGNAL 1: STALENESS VS TREND")

staleness_table = pd.crosstab(
    df["staleness_bucket"],
    df["trend_direction"]
)

print(staleness_table)
print("\nSample count (n):")
print(staleness_table.sum(axis=1))

print("\nRow percentages:")
print((staleness_table.div(staleness_table.sum(axis=1), axis=0) * 100).round(1))

# Signal 2: Search impressions versus observed trend
print("\nSIGNAL 2: IMPRESSIONS VS TREND")

impressions_table = pd.crosstab(
    df["impressions_bucket"],
    df["trend_direction"]
)

print(impressions_table)
print("\nSample count (n):")
print(impressions_table.sum(axis=1))

print("\nRow percentages:")
print((impressions_table.div(impressions_table.sum(axis=1), axis=0) * 100).round(1))

SIGNAL 1: STALENESS VS TREND
trend_direction    down  flat   new  stable    up
staleness_bucket                                 
0-30 days         10473   898  2128    3813  3168
31-90 days          103     1     7      26    38
Over 90 days       5686   253   101    2123  1182

Sample count (n):
staleness_bucket
0-30 days       20480
31-90 days        175
Over 90 days     9345
dtype: int64

Row percentages:
trend_direction   down  flat   new  stable    up
staleness_bucket                                
0-30 days         51.1   4.4  10.4    18.6  15.5
31-90 days        58.9   0.6   4.0    14.9  21.7
Over 90 days      60.8   2.7   1.1    22.7  12.6

SIGNAL 2: IMPRESSIONS VS TREND
trend_direction     down  flat   new  stable    up
impressions_bucket                                
Low                 3116  1116  2026     733  1015
Medium              5115    33   134    1568  1635
High                8031     3    76    3661  1738

Sample count (n):
impressions_bucket
Low        8006
Me

### Signal 1 — Staleness vs Trend

**Verdict: CONFIRMED (directionally).**

The share of pages showing a downward trend increased from 51.1% among pages updated within 30 days to 60.8% among pages not updated for over 90 days. The over-90-day group contained 9,345 pages. This supports staleness as a useful prioritisation signal, but does not establish causation.

### Signal 2 — Impressions vs Trend

**Verdict: CONFIRMED, with a qualification.**

Pages with medium and high 90-day impressions had downward-trend shares of 60.3% and 59.4%, respectively, compared with 38.9% for low-impression pages. This suggests that pages with existing search visibility may deserve review when performance is declining. Medium and high impression groups are similar, so impressions should not be used alone to rank pages.


### My Baseline Scoring Rule

I will use two historical signals to prioritise content pages for human review.

* **Staleness:** Add 1 point if a page has not been updated for more than 90 days.
* **Search visibility:** Add 1 point if a page has received at least 1,000 impressions over the previous 90 days.

The total score ranges from 0 to 2.

**Reason codes:**

* `STALE_AND_VISIBLE` — Both conditions are met.
* `STALE_ONLY` — Only the staleness condition is met.
* `VISIBLE_ONLY` — Only the visibility condition is met.
* `NO_PRIORITY_SIGNAL` — Neither condition is met.

**Action labels:**

* Score 2: `Review for content refresh`
* Score 1: `Secondary review`
* Score 0: `No immediate action`

This rule provides a transparent decision-support ranking. It does not prove that refreshing a page will improve its performance. Human review is required before making content changes.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
import os
import numpy as np
import pandas as pd

# Create a working copy
queue = df.copy()

# Signal 1: Staleness
queue["stale_points"] = (
    queue["days_since_last_update"] > 90
).astype(int)

# Signal 2: Search visibility
queue["visibility_points"] = (
    queue["impressions_90d"] >= 1000
).astype(int)

# Calculate total score
queue["action_score"] = (
    queue["stale_points"] + queue["visibility_points"]
)

# Assign exactly one reason code
queue["reason_code"] = np.select(
    [
        (queue["stale_points"] == 1) & (queue["visibility_points"] == 1),
        (queue["stale_points"] == 1),
        (queue["visibility_points"] == 1)
    ],
    [
        "STALE_AND_VISIBLE",
        "STALE_ONLY",
        "VISIBLE_ONLY"
    ],
    default="NO_PRIORITY_SIGNAL"
)

# Assign action label
queue["action_label"] = np.select(
    [
        queue["action_score"] == 2,
        queue["action_score"] == 1
    ],
    [
        "Review for content refresh",
        "Secondary review"
    ],
    default="No immediate action"
)

# Rank the queue
queue = queue.sort_values(
    by=["action_score", "impressions_90d", "days_since_last_update"],
    ascending=[False, False, False]
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

# Save the ranked queue
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Ranked queue created successfully.")
print("Rows:", len(queue))
print("\nAction score distribution:")
print(queue["action_score"].value_counts().sort_index())

print("\nReason code distribution:")
print(queue["reason_code"].value_counts())

print("\nTop 10 ranked pages:")
print(
    queue[
        ["rank", "action_score", "reason_code",
         "action_label", "days_since_last_update", "impressions_90d"]
    ].head(10).to_string(index=False)
)

print("\nSaved to:", output_path)

Ranked queue created successfully.
Rows: 30000

Action score distribution:
action_score
0    12622
1    11899
2     5479
Name: count, dtype: int64

Reason code distribution:
reason_code
NO_PRIORITY_SIGNAL    12622
VISIBLE_ONLY           8033
STALE_AND_VISIBLE      5479
STALE_ONLY             3866
Name: count, dtype: int64

Top 10 ranked pages:
 rank  action_score       reason_code               action_label  days_since_last_update  impressions_90d
    1             2 STALE_AND_VISIBLE Review for content refresh                     104           517715
    2             2 STALE_AND_VISIBLE Review for content refresh                     104           443434
    3             2 STALE_AND_VISIBLE Review for content refresh                     104           347399
    4             2 STALE_AND_VISIBLE Review for content refresh                     104           309910
    5             2 STALE_AND_VISIBLE Review for content refresh                     104           309192
    6             

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [16]:
# Select the top 20 pages for human review
top20 = queue.head(20).copy()

# Explain why each page was selected
reason_explanations = {
    "STALE_AND_VISIBLE": "Page is over 90 days old and has at least 1,000 impressions.",
    "STALE_ONLY": "Page is over 90 days old but has fewer than 1,000 impressions.",
    "VISIBLE_ONLY": "Page has at least 1,000 impressions but was updated within 90 days.",
    "NO_PRIORITY_SIGNAL": "Page meets neither scoring condition."
}

top20["why_selected"] = top20["reason_code"].map(reason_explanations)

# Add a human-review question for each recommendation
top20["what_could_make_this_wrong"] = (
    "Check whether the page still matches search intent, "
    "whether its information is accurate, and whether its performance "
    "data supports a refresh."
)

# Keep the review table focused and avoid exposing client identifiers
review_columns = [
    "rank",
    "action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "why_selected",
    "what_could_make_this_wrong"
]

top20_review = top20[review_columns].copy()

print("TOP-20 HUMAN REVIEW")
display(top20_review)

TOP-20 HUMAN REVIEW


,rank,action_score,reason_code,action_label,days_since_last_update,impressions_90d,trend_direction,why_selected,what_could_make_this_wrong
0,1,2,STALE_AND_VISIBLE,Review for content refresh,104,517715,down,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
1,2,2,STALE_AND_VISIBLE,Review for content refresh,104,443434,stable,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
2,3,2,STALE_AND_VISIBLE,Review for content refresh,104,347399,down,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
3,4,2,STALE_AND_VISIBLE,Review for content refresh,104,309910,down,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
4,5,2,STALE_AND_VISIBLE,Review for content refresh,104,309192,down,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
5,6,2,STALE_AND_VISIBLE,Review for content refresh,104,295097,stable,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
6,7,2,STALE_AND_VISIBLE,Review for content refresh,104,286608,stable,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
7,8,2,STALE_AND_VISIBLE,Review for content refresh,104,233561,down,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
8,9,2,STALE_AND_VISIBLE,Review for content refresh,104,211366,stable,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...
9,10,2,STALE_AND_VISIBLE,Review for content refresh,104,208678,down,"Page is over 90 days old and has at least 1,00...",Check whether the page still matches search in...


### Top-20 Review Findings

All 20 selected pages received a score of 2 and the `STALE_AND_VISIBLE` reason code. Each page had not been updated for 104 days and had at least 1,000 impressions over the previous 90 days.

However, their observed trends were not uniform:

* 9 pages showed a downward trend.
* 10 pages showed a stable trend.
* 1 page showed an upward trend.

This demonstrates a limitation of the baseline rule. It identifies pages with age and search visibility, but does not distinguish declining pages from stable or improving pages.

The recommendations should therefore be treated as candidates for human review, not automatic refresh decisions. A reviewer should check search intent, content accuracy, and actual performance before approving a refresh.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [18]:
# Identify high-scoring pages that are not showing a downward trend
weak_picks = top20[
    top20["trend_direction"] != "down"
].copy()

print("WEAK PICKS — TOP 20")
print("Number of weak picks:", len(weak_picks))

display(
    weak_picks[
        [
            "rank",
            "action_score",
            "reason_code",
            "days_since_last_update",
            "impressions_90d",
            "trend_direction"
        ]
    ]
)

print("\nTrend distribution in the top 20:")
print(top20["trend_direction"].value_counts())

WEAK PICKS — TOP 20
Number of weak picks: 11


,rank,action_score,reason_code,days_since_last_update,impressions_90d,trend_direction
1,2,2,STALE_AND_VISIBLE,104,443434,stable
5,6,2,STALE_AND_VISIBLE,104,295097,stable
6,7,2,STALE_AND_VISIBLE,104,286608,stable
8,9,2,STALE_AND_VISIBLE,104,211366,stable
10,11,2,STALE_AND_VISIBLE,104,205915,stable
11,12,2,STALE_AND_VISIBLE,104,201584,stable
12,13,2,STALE_AND_VISIBLE,104,201111,stable
13,14,2,STALE_AND_VISIBLE,104,192205,stable
16,17,2,STALE_AND_VISIBLE,104,181574,stable
17,18,2,STALE_AND_VISIBLE,104,181514,up



Trend distribution in the top 20:
trend_direction
stable    10
down       9
up         1
Name: count, dtype: int64


In [19]:
# Verify scoring inputs
scoring_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

print("SCORING INPUTS:")
print(scoring_inputs)

# Check for potential future or target-related columns
suspicious_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in
           ["future", "target", "label", "outcome", "april"])
]

print("\nPotential future or target-related columns:")
print(suspicious_columns)

# Recalculate scores using the sorted queue's own rows
expected_score = (
    (queue["days_since_last_update"] > 90).astype(int)
    + (queue["impressions_90d"] >= 1000).astype(int)
)

print("\nDoes the recalculated score match the ranked queue?")
print(np.array_equal(
    expected_score.to_numpy(),
    queue["action_score"].to_numpy()
))

SCORING INPUTS:
['days_since_last_update', 'impressions_90d']

Potential future or target-related columns:
[]

Does the recalculated score match the ranked queue?
True


### Weak-Pick Review

The top-20 review identified 11 weak picks: 10 pages with stable trends and 1 page with an upward trend. Only 9 of the 20 pages showed a downward trend.

This reveals a limitation of the baseline scoring rule. Staleness and search visibility identify potential review opportunities, but they do not establish that a page is losing performance or needs a refresh.

Human review should check the page's current performance, search intent, and content accuracy before action is taken.


### Leakage and Input Check

The baseline scoring rule uses only two historical inputs: `days_since_last_update` and `impressions_90d`.

The observed `trend_direction` field is used for signal analysis and human review, not as an input to the scoring formula.

The preliminary column-name check found no columns containing future, target, label, outcome, or April-related terms. The independent score recalculation matched the ranked queue (`True`).

This supports the consistency of the scoring implementation. The rule does not intentionally use future outcomes or target-derived inputs.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.